<div dir="rtl" style="text-align: right;">

# بنام خدا

## آزمون پایان دورهٔ تابستانهٔ دومین دورهٔ المپیاد هوش مصنوعی سال ۱۴۰۵

**نام:** ........................................

**نام خانوادگی:** ........................................

</div>


<div dir="rtl" style="text-align: right;">

# 📓 چشم‌های تازه در جنگل


## 1️⃣ شرح مسئله واقعی

یک سازمان محیط‌زیستی ۲۵ دوربینِ تله‌ای (camera trap) در یک منطقه‌ی حفاظت‌شده نصب کرده تا از روی هر عکسِ گرفته‌شده، یکی از ۵ گزینه را تشخیص دهد: `deer`، `fox`، `boar`، `owl`، یا `empty` (قاب خالی). هر عکس با متادیتای زیر همراه است: `location_id` (کدام یک از دوربین‌های ثابت)، `time_of_day` (ساعت، پیوسته)، `ambient_temp_c` (دمای محیط).

هر دوربین روی درختی نصب شده که **بر اساس زیستگاهِ شناخته‌شده‌ی همان گونه** انتخاب شده (مثلاً جغدها نزدیک کاج‌های بلند، آهوها نزدیک چمنزار). علاوه‌بر این، حدود ۲۰٪ رکوردهای آزمون به‌خاطر یک باگ نرم‌افزاریِ دوربین‌های تازه، `time_of_day` یا `ambient_temp_c` گم‌شده (`NaN`) دارند.

## 2️⃣ شناسنامه‌ی داده

| فایل | کلیدها |
| --- | --- |
| `train_traps.npz` | `images` (۳۰۰۰×۴۸×۴۸)، `location_id`، `time_of_day`، `ambient_temp_c`، `label` (۰ تا ۴)، `class_names` |
| `test_traps_unlabeled.npz` | `images` (۹۰۰×۴۸×۴۸)، `location_id` (مقادیرِ تازه، هرگز در train دیده نشده)، `time_of_day` و `ambient_temp_c` (در ~۲۰٪ رکوردها دست‌کم یکی از این دو `NaN` است)، `class_names` |

</div>


<div dir="rtl" style="text-align: right;">

## 3️⃣ شیوه‌ی نمره‌دهی

</div>



$$
\text{Score} =
\begin{cases}
0 & \text{Macro-F1}_{\text{test}} < 0.65 \\
25 + 75 \times \dfrac{\text{Macro-F1}_{\text{test}} - 0.65}{0.90 - 0.65} & 0.65 \le \text{Macro-F1}_{\text{test}} < 0.90 \\
100 & \text{Macro-F1}_{\text{test}} \ge 0.90
\end{cases}
$$


<div dir="rtl" style="text-align: right;">

## 4️⃣ قرارداد فنی (الزامات)

- کتابخانه‌های مجاز: `numpy`, `pandas`, `torch`، `matplotlib`، `PIL`/`scipy.ndimage`. `scikit-learn` فقط برای معیارها.
- **بازبینی دستی:** مدل نهایی باید **هر دو وجه (تصویر و متادیتا)** را واقعاً در تصمیم نهایی دخیل کند.
- پیش‌بینی نهایی روی داده‌ی آزمون باید در متغیر `final_pred` (آرایه‌ی numpy یک‌بعدی به طول ۹۰۰ با مقادیر ۰ تا ۴) ذخیره شود.
- خروجی نهایی: فایل `submission.zip` (شامل `submission.csv` با ستون‌های `id` و `label` و همین نوت‌بوک) که سلول آخر می‌سازد.

</div>


<div dir="rtl" style="text-align: right;">

## 5️⃣ بارگذاری داده‌ها و بررسی اولیه

</div>


In [ ]:
train_npz = "train_traps.npz"
test_npz = "test_traps_unlabeled.npz"
output_csv = "submission.csv"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

tr = np.load(train_npz, allow_pickle=True)
images, location_id, time_of_day, ambient_temp_c, label = (
    tr["images"], tr["location_id"], tr["time_of_day"], tr["ambient_temp_c"], tr["label"])
class_names = [str(c) for c in tr["class_names"]]

te = np.load(test_npz, allow_pickle=True)
test_images = te["images"]
test_location_id, test_time_of_day, test_ambient_temp_c = (
    te["location_id"], te["time_of_day"], te["ambient_temp_c"])

print("train images:", images.shape, "labels:", np.bincount(label))
print("test images:", test_images.shape)
print("class_names:", class_names)
print("train locations:", sorted(set(location_id.tolist())))
print("test locations (new!):", sorted(set(test_location_id.tolist())))
print("missing time_of_day in test:", np.isnan(test_time_of_day).sum(),
      "| missing ambient_temp_c in test:", np.isnan(test_ambient_temp_c).sum())

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for c in range(5):
    i = np.where(label == c)[0][0]
    axes[c].imshow(images[i], cmap="gray", vmin=0, vmax=255)
    axes[c].set_title(class_names[c])
    axes[c].axis("off")
plt.tight_layout(); plt.show()

<div dir="rtl" style="text-align: right;">

## 6️⃣ تحلیل اکتشافی داده (EDA)

داده‌ها را بررسی کنید: توزیع هر ویژگی (تصویر و متادیتا)، وابستگی ویژگی‌ها به یکدیگر و به برچسب، و هر چیزی که فکر می‌کنید روی طراحی مدل اثر می‌گذارد. یافته‌های مهم خود را کوتاه در یک سلول متنی بنویسید.

</div>


In [ ]:
# Explore the data (distributions, dependencies between features and with the label, ...).
raise NotImplementedError("Complete the EDA")

<div dir="rtl" style="text-align: right;">

## 7️⃣ مدل تلفیقی (تصویر + متادیتا)

یک مدل بسازید که متادیتا را با تصویر ترکیب کند و از روی هر دو، برچسب را پیش‌بینی کند. `FusionNet`، `train_fusion` و `predict_fusion` را کامل کنید، مدل را اعتبارسنجی کنید و در پایان پیش‌بینی مدل روی داده‌ی آزمون را در `final_pred` ذخیره کنید.

</div>


In [ ]:
def macro_f1(y_true, y_pred, n_classes=5):
    f1s = []
    for c in range(n_classes):
        tp = np.sum((y_pred == c) & (y_true == c))
        fp = np.sum((y_pred == c) & (y_true != c))
        fn = np.sum((y_pred != c) & (y_true == c))
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        f1s.append(f1)
    return float(np.mean(f1s))


class FusionNet(nn.Module):
    def __init__(self, meta_dim, n_classes=5):
        super().__init__()
        # ==============================================================================
        # Build an image branch (multi-layer Conv2d -> embedding) and a metadata
        # branch (multi-layer Linear -> embedding), concatenate the two embeddings
        # (torch.cat), and map them to n_classes with a classification head.
        # ==============================================================================
        raise NotImplementedError("FusionNet.__init__ is not implemented")

    def forward(self, x_img, x_meta):
        raise NotImplementedError("FusionNet.forward is not implemented")


def img_tensor(imgs):
    return torch.tensor(imgs.astype(np.float32) / 255.0, dtype=torch.float32).unsqueeze(1)


def train_fusion(model, imgs, meta, y, epochs=14, lr=1e-3, batch_size=64, seed=0):
    # Train `model` on (imgs, meta) -> y and return the trained model.
    raise NotImplementedError("train_fusion is not implemented")


def predict_fusion(model, imgs, meta):
    # Return the predicted class labels (np.ndarray of shape (N,)) for (imgs, meta).
    raise NotImplementedError("predict_fusion is not implemented")

# Build the metadata features, train the fusion model, validate it, and store the
# test-set predictions (np.ndarray of shape (900,), values 0..4) in `final_pred`.


<div dir="rtl" style="text-align: right;">

## 8️⃣ ساخت فایل نهایی تحویل

پیش از اجرای سلول آخر، نوت‌بوک تکمیل‌شده را با نام `question.ipynb` در همین پوشه ذخیره کنید.

سلول آخر (از متغیر `final_pred` در بخش 7️⃣ استفاده می‌کند):

- پیش‌بینی‌ها را بررسی می‌کند؛
- فایل `submission.csv` را با ستون‌های `id` و `label` می‌سازد؛
- فایل `submission.zip` را می‌سازد که دقیقاً شامل `submission.csv` و همین نوت‌بوک تکمیل‌شده است.

فایل `submission.zip` را بارگذاری کنید، نه فایل CSV را به‌تنهایی.

</div>


In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

NOTEBOOK_FILENAME = "question.ipynb"
CSV_FILENAME = "submission.csv"
ZIP_FILENAME = "submission.zip"

predictions = np.asarray(final_pred)
if predictions.ndim != 1:
    raise ValueError("Predictions must be one-dimensional.")
if len(predictions) != len(test_images):
    raise ValueError("Prediction count does not match test row count.")
if pd.isna(predictions).any() or not np.isfinite(predictions).all():
    raise ValueError("Predictions contain invalid values.")
if not np.equal(predictions, predictions.astype(int)).all():
    raise ValueError("Predictions must be integer class labels.")

predictions = predictions.astype(int)
if not np.isin(predictions, [0, 1, 2, 3, 4]).all():
    raise ValueError("Every prediction must be one of 0, 1, 2, 3, 4.")

notebook_path = Path(NOTEBOOK_FILENAME)
if not notebook_path.is_file():
    raise FileNotFoundError(
        f"Save this notebook as {NOTEBOOK_FILENAME} in the current folder, then run this cell again."
    )

submission = pd.DataFrame({
    "id": np.arange(len(test_images), dtype=int),
    "label": predictions,
})
if submission["id"].isna().any() or submission["id"].duplicated().any():
    raise ValueError("Identifier values must be present and unique.")

submission.to_csv(CSV_FILENAME, index=False)
with ZipFile(ZIP_FILENAME, "w", compression=ZIP_DEFLATED) as archive:
    archive.write(CSV_FILENAME, arcname=CSV_FILENAME)
    archive.write(notebook_path, arcname=NOTEBOOK_FILENAME)

print(f"{ZIP_FILENAME} created successfully.")
display(submission.head())
